# 03_preprocessing
2주차 B팀 전처리 코드입니다.


In [ ]:
import pandas as pd
import numpy as np

sale = pd.read_csv("../raw/sale_all.csv", low_memory=False)
rent = pd.read_csv("../raw/rent_all.csv", low_memory=False)

gu_map = {
    11110:'종로구',11140:'중구',11170:'용산구',11200:'성동구',11215:'광진구',
    11230:'동대문구',11260:'중랑구',11290:'성북구',11305:'강북구',11320:'도봉구',
    11350:'노원구',11380:'은평구',11410:'서대문구',11440:'마포구',11470:'양천구',
    11500:'강서구',11530:'구로구',11545:'금천구',11560:'영등포구',11590:'동작구',
    11620:'관악구',11650:'서초구',11680:'강남구',11710:'송파구',11740:'강동구'
}

def num_clean(s):
    return pd.to_numeric(
        s.astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )

def clean_sale(df):
    d = df.copy()
    d['sggCd'] = pd.to_numeric(d['sggCd'], errors='coerce').astype('Int64')
    d['gu'] = d['sggCd'].map(gu_map)
    d['contract_month'] = d['dealYear'].astype(str) + '-' + d['dealMonth'].astype(str).str.zfill(2)
    d['sale_price'] = num_clean(d['dealAmount'])
    d['area_m2'] = pd.to_numeric(d['excluUseAr'], errors='coerce')
    d['sale_price_per_m2'] = d['sale_price'] / d['area_m2']
    d['age'] = pd.to_numeric(d['dealYear'], errors='coerce') - pd.to_numeric(d['buildYear'], errors='coerce')
    return d

def clean_rent(df):
    d = df.copy()
    d['sggCd'] = pd.to_numeric(d['sggCd'], errors='coerce').astype('Int64')
    d['gu'] = d['sggCd'].map(gu_map)
    d['contract_month'] = d['dealYear'].astype(str) + '-' + d['dealMonth'].astype(str).str.zfill(2)
    d['deposit_num'] = num_clean(d['deposit'])
    d['monthly_rent_num'] = num_clean(d['monthlyRent'])
    d['area_m2'] = pd.to_numeric(d['excluUseAr'], errors='coerce')
    d['deposit_per_m2'] = d['deposit_num'] / d['area_m2']
    d['rent_type'] = np.where(d['monthly_rent_num'].fillna(0) == 0, 'jeonse', 'monthly')
    d['age'] = pd.to_numeric(d['dealYear'], errors='coerce') - pd.to_numeric(d['buildYear'], errors='coerce')
    return d

sale_clean = clean_sale(sale)
rent_clean = clean_rent(rent)

sale_monthly = sale_clean.groupby(['sggCd','gu','contract_month'], dropna=False).agg(
    avg_sale_price=('sale_price','mean'),
    med_sale_price=('sale_price','median'),
    avg_sale_price_per_m2=('sale_price_per_m2','mean'),
    med_sale_price_per_m2=('sale_price_per_m2','median'),
    sale_count=('sale_price','size')
).reset_index()

jeonse_monthly = rent_clean[rent_clean['rent_type'] == 'jeonse'].groupby(['sggCd','gu','contract_month'], dropna=False).agg(
    avg_jeonse_deposit=('deposit_num','mean'),
    med_jeonse_deposit=('deposit_num','median'),
    avg_jeonse_deposit_per_m2=('deposit_per_m2','mean'),
    med_jeonse_deposit_per_m2=('deposit_per_m2','median'),
    jeonse_count=('deposit_num','size')
).reset_index()

rent_monthly = rent_clean.groupby(['sggCd','gu','contract_month'], dropna=False).agg(
    total_rent_count=('deposit_num','size'),
    monthly_count=('rent_type', lambda x: int((x == 'monthly').sum())),
    jeonse_count_from_rent=('rent_type', lambda x: int((x == 'jeonse').sum()))
).reset_index()

rent_monthly['monthly_ratio'] = rent_monthly['monthly_count'] / rent_monthly['total_rent_count']

monthly_merged = sale_monthly.merge(
    jeonse_monthly,
    on=['sggCd','gu','contract_month'],
    how='inner'
).merge(
    rent_monthly[['sggCd','gu','contract_month','total_rent_count','monthly_count','monthly_ratio']],
    on=['sggCd','gu','contract_month'],
    how='left'
)

monthly_merged = monthly_merged.sort_values(['sggCd','contract_month']).reset_index(drop=True)

monthly_merged['jeonse_rate'] = monthly_merged['avg_jeonse_deposit_per_m2'] / monthly_merged['avg_sale_price_per_m2']
monthly_merged['gap_rate'] = 1 - monthly_merged['jeonse_rate']

monthly_merged['sale_growth_1m'] = monthly_merged.groupby('sggCd')['avg_sale_price_per_m2'].pct_change()
monthly_merged['jeonse_growth_1m'] = monthly_merged.groupby('sggCd')['avg_jeonse_deposit_per_m2'].pct_change()
monthly_merged['growth_gap_1m'] = monthly_merged['sale_growth_1m'] - monthly_merged['jeonse_growth_1m']
monthly_merged['sale_volume_growth_1m'] = monthly_merged.groupby('sggCd')['sale_count'].pct_change()
monthly_merged['rent_volume_growth_1m'] = monthly_merged.groupby('sggCd')['total_rent_count'].pct_change()

monthly_merged.to_csv("../processed/monthly_merged.csv", index=False, encoding="utf-8-sig")


OSError: Cannot save file into a non-existent directory: 'data/processed'